In [3]:
import xgboost as xgb
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET].map({"No":0, "Yes":1})
X = pd.get_dummies(train.drop(columns=[TARGET]))
test = pd.get_dummies(test)

test = test.reindex(columns=X.columns, fill_value=0)

"""model = xgb.XGBClassifier(
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    eval_metric="auc",
    n_estimators=3000,
    max_depth=8,
    learning_rate=0.03
)

model.fit(X, y)"""

model = xgb.XGBClassifier(
    tree_method="hist",
    device="cuda",
    eval_metric="auc",
    n_estimators=3000,
    max_depth=8,
    learning_rate=0.03
)

model.fit(X, y)

pred_probs = model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_xgb_gpu.csv", index=False)

/home/chotu/.virtualenvs/torch/lib/python3.13/site-packages/xgboost/core.py:751: UserWarning: [17:52:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [2]:
import xgboost
print(xgboost.__version__)
print(xgboost.build_info())

3.2.0
{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [12, 9], 'DEBUG': False, 'GCC_VERSION': [10, 3, 1], 'GLIBC_VERSION': [2, 28], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 29, 2], 'THRUST_VERSION': [2, 8, 2], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': True, 'USE_FEDERATED': True, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/home/chotu/.virtualenvs/torch/lib/python3.13/site-packages/xgboost/lib/libxgboost.so'}
